# NB_Raw_Ingest
Fetches **one resource for one day** from the FHIR API with pagination and stores every page **exactly as received** in
`Files/raw/<Resource>/ingest_date=<load_date>/page_NNNNN.json`. Logs every API call to `meta.pipeline_log`.
Called by the pipeline once per resource; the pipeline supplies `resource_name`, `load_date`, `run_id`.

In [77]:
resource_name = "Condition"       # Patient / Encounter / Observation / Condition
load_date     = "2026-11-04"    # day to load (YYYY-MM-DD). Window = [load_date 00:00Z, next day 00:00Z)
run_id = ""
max_pages     = 2000            # safety cap; the run FAILS if the cap is hit (never silently incomplete)

StatementMeta(, 5189a734-268d-4f81-b22f-d4e34d2c875d, 79, Finished, Available, Finished, False)

In [ ]:

import os, time, uuid
import requests
from delta.tables import DeltaTable
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as F
from pyspark.sql import types as T

run_id = run_id or str(uuid.uuid4())   # pipeline passes @pipeline().RunId; otherwise generate one

raw_root = "/lakehouse/default/Files/raw"

def now_utc():
    return datetime.now(timezone.utc).replace(tzinfo=None)

datetime.strptime(load_date, "%Y-%m-%d")

print("Resource:", resource_name)
print("Load date:", load_date)
print("Run ID:", run_id)
print("Maximum pages:", max_pages)


In [79]:

spark.sql("""
CREATE TABLE IF NOT EXISTS meta.resource_watermark (
    resource_name STRING,
    last_loaded_date STRING,
    updated_at TIMESTAMP
)
USING DELTA
""")

print("Watermark table ready")
spark.table("meta.resource_watermark").printSchema()
display(spark.table("meta.resource_watermark"))


StatementMeta(, 5189a734-268d-4f81-b22f-d4e34d2c875d, 81, Finished, Available, Finished, False)

Watermark table ready
root
 |-- resource_name: string (nullable = true)
 |-- last_loaded_date: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)



SynapseWidget(Synapse.DataFrame, 82c80c38-2581-4a36-bfc6-c9aaa84ae7c7)

In [80]:
display(
    spark.table("meta.config_resources")
    .filter(F.col("resource_name") == resource_name)
)

StatementMeta(, 5189a734-268d-4f81-b22f-d4e34d2c875d, 82, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4bdc8c64-fa31-43fc-be70-1d3a7c712de7)

In [81]:
display(
    spark.table("meta.resource_watermark")
    .filter(F.col("resource_name") == resource_name)
)

StatementMeta(, 5189a734-268d-4f81-b22f-d4e34d2c875d, 83, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ca8c47c8-4c53-445f-8049-71fe76e4265d)

In [82]:

required_tables = [
    "meta.config_resources",
    "meta.pipeline_log",
    "meta.resource_watermark"
]

for table_name in required_tables:
    if not spark.catalog.tableExists(table_name):
        raise ValueError(f"Missing table: {table_name}")
    print(f"PASS: {table_name}")

print("\nResource configuration:")
display(spark.table("meta.config_resources"))

print("\nWatermark:")
display(spark.table("meta.resource_watermark"))


StatementMeta(, 5189a734-268d-4f81-b22f-d4e34d2c875d, 84, Finished, Available, Finished, False)

PASS: meta.config_resources
PASS: meta.pipeline_log
PASS: meta.resource_watermark

Resource configuration:


SynapseWidget(Synapse.DataFrame, 4ea22dc4-ceac-4ab0-8be9-fbc997535a98)


Watermark:


SynapseWidget(Synapse.DataFrame, 44cbad49-f0fc-46f2-84e6-55d434d9de7b)

In [83]:

existing_columns = set(spark.table("meta.pipeline_log").columns)

required_columns = {
    "load_date": "STRING",
    "step": "STRING",
    "rows_written": "BIGINT",
    "api_total": "BIGINT"
}

for column_name, column_type in required_columns.items():
    if column_name not in existing_columns:
        spark.sql(f"""
            ALTER TABLE meta.pipeline_log
            ADD COLUMNS ({column_name} {column_type})
        """)
        print(f"Added: {column_name}")
    else:
        print(f"Already exists: {column_name}")

spark.table("meta.pipeline_log").printSchema()


StatementMeta(, 5189a734-268d-4f81-b22f-d4e34d2c875d, 85, Finished, Available, Finished, False)

Already exists: load_date
Already exists: step
Already exists: rows_written
Already exists: api_total
root
 |-- run_id: string (nullable = true)
 |-- resource_name: string (nullable = true)
 |-- layer: string (nullable = true)
 |-- page_no: integer (nullable = true)
 |-- api_url: string (nullable = true)
 |-- api_call_time: timestamp (nullable = true)
 |-- http_status: integer (nullable = true)
 |-- rows_fetched: integer (nullable = true)
 |-- raw_file_path: string (nullable = true)
 |-- data_saved_time: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- error_message: string (nullable = true)
 |-- load_date: string (nullable = true)
 |-- step: string (nullable = true)
 |-- rows_written: long (nullable = true)
 |-- api_total: long (nullable = true)



In [84]:
cfg = (
    spark.table("meta.config_resources")
    .filter(
        (F.col("resource_name") == resource_name) &
        (F.col("is_active") == True)
    )
    .first()
)

if cfg is None:
    raise ValueError(
        f"{resource_name} is not active in meta.config_resources"
    )

base_url = cfg["endpoint"]
page_size = cfg["page_size"]

print("Endpoint:", base_url)
print("Page size:", page_size)

StatementMeta(, 5189a734-268d-4f81-b22f-d4e34d2c875d, 86, Finished, Available, Finished, False)

Endpoint: https://hapi.fhir.org/baseR4
Page size: 100


In [85]:
day = datetime.strptime(load_date, "%Y-%m-%d")

from_ts = day.strftime("%Y-%m-%dT00:00:00Z")
to_ts = (day + timedelta(days=1)).strftime("%Y-%m-%dT00:00:00Z")

window = [
    ("_lastUpdated", f"ge{from_ts}"),
    ("_lastUpdated", f"lt{to_ts}")
]

print("From:", from_ts)
print("To:", to_ts)

StatementMeta(, 5189a734-268d-4f81-b22f-d4e34d2c875d, 87, Finished, Available, Finished, False)

From: 2026-11-03T00:00:00Z
To: 2026-11-04T00:00:00Z


In [86]:

# ==================================================
# STEP 8: API RETRY FUNCTION
# ==================================================

import requests
import time

def get_with_retry(url, params=None, retries=5, backoff=5):

    last_error = None

    for attempt in range(1, retries + 1):

        try:
            response = requests.get(
                url,
                params=params,
                headers={
                    "Accept": "application/fhir+json"
                },
                timeout=120
            )

            if response.status_code == 200:
                return response

            if response.status_code not in (
                429, 500, 502, 503, 504
            ):
                response.raise_for_status()

            last_error = f"HTTP {response.status_code}"

        except requests.RequestException as e:
            last_error = str(e)

        if attempt < retries:
            wait_time = backoff * attempt
            print(
                f"Attempt {attempt} failed: {last_error}. "
                f"Retrying in {wait_time} seconds..."
            )
            time.sleep(wait_time)

    raise RuntimeError(
        f"API request failed after {retries} attempts: {last_error}"
    )

print("API retry function ready.")


StatementMeta(, 5189a734-268d-4f81-b22f-d4e34d2c875d, 88, Finished, Available, Finished, False)

API retry function ready.


In [87]:
api_total = get_with_retry(
    f"{base_url}/{resource_name}",
    window + [
        ("_summary", "count"),
        ("_total", "accurate")
    ]
).json().get("total")

print(f"{resource_name} {load_date}: API total = {api_total}")
print("Run ID:", run_id)

StatementMeta(, 5189a734-268d-4f81-b22f-d4e34d2c875d, 89, Finished, Available, Finished, False)

Condition 2026-11-03: API total = 989
Run ID: 70940515-f540-4403-a7c2-17e2a72e89d6


In [88]:

# --------------------------------------------------
# 1. METADATA LOGGING FUNCTIONS (SCHEMA-SAFE)
# --------------------------------------------------

from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, LongType, TimestampType
)

LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("resource_name", StringType(), True),
    StructField("load_date", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("step", StringType(), True),
    StructField("page_no", IntegerType(), True),
    StructField("api_url", StringType(), True),
    StructField("api_call_time", TimestampType(), True),
    StructField("http_status", IntegerType(), True),
    StructField("rows_fetched", IntegerType(), True),
    StructField("rows_written", LongType(), True),
    StructField("api_total", LongType(), True),
    StructField("raw_file_path", StringType(), True),
    StructField("data_saved_time", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])


def log_rows(rows):

    if not rows:
        return

    log_df = spark.createDataFrame(
        rows,
        schema=LOG_SCHEMA
    )

    target_schema = spark.table(
        "meta.pipeline_log"
    ).schema

    target_columns = [
        field.name for field in target_schema.fields
    ]

    # Match every column to the existing Delta table type
    for field in target_schema.fields:
        if field.name in log_df.columns:
            log_df = log_df.withColumn(
                field.name,
                F.col(field.name).cast(field.dataType)
            )

    # Add any missing target columns as NULL
    for field in target_schema.fields:
        if field.name not in log_df.columns:
            log_df = log_df.withColumn(
                field.name,
                F.lit(None).cast(field.dataType)
            )

    log_df = log_df.select(*target_columns)

    log_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("meta.pipeline_log")


def log_summary(
    layer,
    rows_fetched,
    rows_written,
    status,
    message,
    api_total,
    api_url,
    file_path
):

    summary = [(
        run_id,
        resource_name,
        load_date,
        layer,
        "summary",
        None,
        api_url,
        now_utc(),
        None,
        rows_fetched,
        rows_written,
        api_total,
        file_path,
        now_utc(),
        status,
        message
    )]

    log_rows(summary)


StatementMeta(, 5189a734-268d-4f81-b22f-d4e34d2c875d, 90, Finished, Available, Finished, False)

In [89]:
spark.table("meta.pipeline_log").printSchema()

StatementMeta(, 5189a734-268d-4f81-b22f-d4e34d2c875d, 91, Finished, Available, Finished, False)

root
 |-- run_id: string (nullable = true)
 |-- resource_name: string (nullable = true)
 |-- layer: string (nullable = true)
 |-- page_no: integer (nullable = true)
 |-- api_url: string (nullable = true)
 |-- api_call_time: timestamp (nullable = true)
 |-- http_status: integer (nullable = true)
 |-- rows_fetched: integer (nullable = true)
 |-- raw_file_path: string (nullable = true)
 |-- data_saved_time: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- error_message: string (nullable = true)
 |-- load_date: string (nullable = true)
 |-- step: string (nullable = true)
 |-- rows_written: long (nullable = true)
 |-- api_total: long (nullable = true)



In [ ]:
# ==================================================
# PAGINATE THE API AND STORE EVERY PAGE EXACTLY AS RECEIVED
# ==================================================
# Path:  Files/raw/<Resource>/ingest_date=<load_date>/run_id=<run_id>/page_NNNNN.json
rel_dir = f"Files/raw/{resource_name}/ingest_date={load_date}/run_id={run_id}"
out_dir = f"{raw_root}/{resource_name}/ingest_date={load_date}/run_id={run_id}"
os.makedirs(out_dir, exist_ok=True)

url = f"{base_url}/{resource_name}"
params = window + [("_count", page_size), ("_sort", "_lastUpdated")]

page_logs = []
page_no, rows, status, message = 0, 0, "SUCCESS", None
first_url = None

def next_link(bundle):
    for link in bundle.get("link", []):
        if link.get("relation") == "next":
            return link.get("url")
    return None

try:
    while url:
        if page_no >= max_pages:
            raise RuntimeError(
                f"max_pages ({max_pages}) reached before the last page; "
                "extraction would be incomplete"
            )
        page_no += 1
        call_time = now_utc()
        resp = get_with_retry(url, params if page_no == 1 else None)
        bundle = resp.json()
        n = len(bundle.get("entry", []))

        file_name = f"page_{page_no:05d}.json"
        with open(f"{out_dir}/{file_name}", "w", encoding="utf-8") as fh:
            fh.write(resp.text)                      # stored exactly as received

        if first_url is None:
            first_url = resp.url
        rows += n
        page_logs.append((
            run_id, resource_name, load_date, "raw", "page", page_no, resp.url,
            call_time, resp.status_code, n, n, api_total,
            f"{rel_dir}/{file_name}", now_utc(), "SUCCESS", None
        ))
        url = next_link(bundle)
        time.sleep(0.5)                              # be polite to the public server

except Exception as e:
    status, message = "FAILED", str(e)[:1000]
    page_logs.append((
        run_id, resource_name, load_date, "raw", "page", page_no, url,
        now_utc(), None, 0, 0, api_total, None, now_utc(), "FAILED", message
    ))

if status == "SUCCESS" and api_total is not None and rows != api_total:
    status = "WARN"
    message = f"Fetched {rows} rows but the API reported {api_total}"

log_rows(page_logs)
log_summary(
    layer="raw",
    rows_fetched=rows,
    rows_written=rows,
    status=status,
    message=message or f"{resource_name} {load_date}: {page_no} pages saved",
    api_total=api_total,
    api_url=first_url,
    file_path=rel_dir
)

print(f"{resource_name} {load_date}: {page_no} pages, {rows} rows, status={status}")
print("Raw directory:", out_dir)

if status == "FAILED":
    raise RuntimeError(f"Raw ingestion failed for {resource_name}: {message}")

In [90]:

# Update watermark only after successful count reconciliation

from delta.tables import DeltaTable

if status == "SUCCESS" and api_total is not None and api_total == rows:

    wm = spark.createDataFrame(
        [(resource_name, load_date, now_utc())],
        """
        resource_name STRING,
        last_loaded_date STRING,
        updated_at TIMESTAMP
        """
    )

    target = DeltaTable.forName(
        spark, "meta.resource_watermark"
    )

    (
        target.alias("t")
        .merge(
            wm.alias("s"),
            "t.resource_name = s.resource_name"
        )
        .whenMatchedUpdate(set={
            "last_loaded_date": "s.last_loaded_date",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsert(values={
            "resource_name": "s.resource_name",
            "last_loaded_date": "s.last_loaded_date",
            "updated_at": "s.updated_at"
        })
        .execute()
    )

    print("Watermark updated successfully.")

else:
    print("Watermark not updated: extraction not fully reconciled.")

display(spark.table("meta.resource_watermark"))


StatementMeta(, 5189a734-268d-4f81-b22f-d4e34d2c875d, 92, Finished, Available, Finished, False)

Watermark updated successfully.


SynapseWidget(Synapse.DataFrame, 638198ab-53da-41ae-b3ea-b82deb474526)

In [92]:
display(
    spark.table("meta.pipeline_log")
    .filter(
        (F.col("resource_name") == resource_name) &
        (F.col("load_date") == load_date) &
        (F.col("layer") == "raw") &
        (F.col("step") == "summary")
    )
    .orderBy(F.col("data_saved_time").desc())
)

StatementMeta(, 5189a734-268d-4f81-b22f-d4e34d2c875d, 94, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e7d91911-fc4c-4105-bc75-922c897bd695)

In [93]:
display(
    spark.table("meta.resource_watermark")
    .filter(F.col("resource_name") == resource_name)
)

StatementMeta(, 5189a734-268d-4f81-b22f-d4e34d2c875d, 95, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a38a4d9f-ff20-461a-bdc0-511dd691bf16)